Since there is a retraining needed for poor accuracy values. So to hold up the FGP masks we need a interactive environment to retain and experiment with diffrent parameters.

In [ ]:

import sys
import torch
from torch import nn
import copy
import random
import torch
from pathlib import Path

# Notebook is inside .../PruningNAS/PruningNAS, so add project root (.../PruningNAS)
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.insert(0, str(project_root))

# Install local package in editable mode (run once; safe to re-run)
%pip install -e ..


from PruningNAS.DataProcess.DataPreprocessing import get_dataloaders
from PruningNAS.Utills.EvaluatiorUtills import get_model_size, get_sparsity
from PruningNAS.Utills.PrunUtillCP import ChannelPrunner
from PruningNAS.Utills.PrunUtillFGP import FineGrainedPruner
from PruningNAS.Utills.TrainingModulesUtills import TrainingPrunned, evaluate
from PruningNAS.Utills.Utill import print_model
from PruningNAS.Utills.ViewerUtills import plot_accuracy, plot_loss  # Ensure you import your correct model architecture
seed=0
random.seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Byte = 8
KiB = 1024 * Byte
MiB = 1024 * KiB
GiB = 1024 * MiB

Set Static Params here:

In [ ]:
import os


current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")


In [ ]:

# Initialize the model
basedir=''
path='./dataset/cifar10'
select_model='MobilenetV2'
pruning_type='CP'
#model_path='./checkpoint/vgg_mrl_99.51375579833984.pth'
model_path=r'.\checkpoint\MobilenetV2\MobilenetV2_cifar_95.029999.pth'
# Load the saved state_dict correctly
model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

model.to(device)

sparsity_dict  = {
"features.0.0": 0.40,
"features.1.block.0": 0.50,
"features.1.block.3": 0.60,
"features.2.block.0": 0.60,
"features.2.block.3": 0.60,
"features.2.block.6": 0.50,
"features.3.block.0": 0.80,
"features.3.block.3": 0.70,
"features.3.block.6": 0.70,
"features.4.block.0": 0.60,
"features.4.block.3": 0.30,
"features.4.block.6": 0.50,
"features.5.block.0": 0.80,
"features.5.block.3": 0.70,
"features.5.block.6": 0.80,
"features.6.block.0": 0.80,
"features.6.block.3": 0.40,
"features.6.block.6": 0.80,
"features.7.block.0": 0.60,
"features.7.block.3": 0.40,
"features.7.block.6": 0.60,
"features.8.block.0": 0.90,
"features.8.block.3": 0.90,
"features.8.block.6": 0.90,
"features.9.block.0": 0.90,
"features.9.block.3": 0.90,
"features.9.block.6": 0.90,
"features.10.block.0": 0.90,
"features.10.block.3": 0.90,
"features.10.block.6": 0.90,
"features.11.block.0": 0.40,
"features.11.block.3": 0.50,
"features.11.block.6": 0.40,
"features.12.block.0": 0.90,
"features.12.block.3": 0.90,
"features.12.block.6": 0.90,
"features.13.block.0": 0.90,
"features.13.block.3": 0.90,
"features.13.block.6": 0.90,
"features.14.block.0": 0.50,
"features.14.block.3": 0.50,
"features.14.block.6": 0.70,
"features.15.block.0": 0.90,
"features.15.block.3": 0.70,
"features.15.block.6": 0.90,
"features.16.block.0": 0.80,
"features.16.block.3": 0.70,
"features.16.block.6": 0.80,
"features.17.block.0": 0.90,
"features.17.block.3": 0.90,
"features.17.block.6": 0.90,
"features.18": 0.90,
"fc": 0.90
}


# {
# "model.0": 0.1,
# "model.3.depthwise": 0.2,
# "model.4.depthwise": 0.1,
# "model.5.depthwise": 0.1,
# "model.6.depthwise": 0.2,
# "model.7.depthwise": 0.3,
# "model.8.depthwise": 0.2,
# "model.9.depthwise": 0.3,
# "model.10.depthwise": 0.2,
# "model.11.depthwise": 0.3,
# "model.12.depthwise": 0.5,
# "model.13.depthwise": 0.4,
# "model.14.depthwise": 0.7,
# "model.15.depthwise": 0.8
# }


Define experimental params here:

In [ ]:
num_finetune_epochs = 300
lr=0.01

In [ ]:
train_dataloader,test_dataloader=get_dataloaders(path, batch_size=256 ) # Basemodel
dense_model_accuracy=evaluate(model,test_dataloader)
print('dense_model_accuracy:',dense_model_accuracy)


In [ ]:
pruned_model=copy.deepcopy(model)
pruned_model.requires_grad_(False)  # Disable gradients before pruning

if pruning_type=='FGP':
    isCallback=True
    pruner = FineGrainedPruner(pruned_model, sparsity_dict)
elif pruning_type=='CP':
    pruned_model=ChannelPrunner(pruned_model, sparsity_dict,select_model)
    pruner=None
    isCallback=False
else:
    print('pruning_type doesn\'t exists')
    exit

print_model(pruned_model)
print(f'The sparsity of each layer becomes')
for name, param in pruned_model.named_parameters():
    print(f'  {name}: {get_sparsity(param):.2f}')


In [ ]:
# model_path=r'.\checkpoint\Densenet-121\CP\Densenet-121_cifar_CP_94.58999633789062.pth'
# # Load the saved state_dict correctly
# pruned_model = torch.load(model_path, map_location=torch.device(device),weights_only=False)  # Use 'cpu' if necessary

# pruned_model.to(device)

In [ ]:

dense_model_size = get_model_size(model, count_nonzero_only=True)
sparse_model_size = get_model_size(pruned_model, count_nonzero_only=True)

print(f"Sparse model has size={sparse_model_size:.2f} MiB = {sparse_model_size / dense_model_size * 100:.2f}% of dense model size")
sparse_model_accuracy,_ = evaluate(pruned_model, test_dataloader)
print(f"Sparse model has accuracy={sparse_model_accuracy:.2f}% before fintuning")

# Re-enable gradients for training
pruned_model.requires_grad_(True)

lr
optimizer = torch.optim.SGD(pruned_model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, num_finetune_epochs)
criterion = nn.CrossEntropyLoss()


In [ ]:
pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)
print(pruned_model_accuracy)


In [ ]:
print(pruned_model_accuracy)
basedir='.'

torch.save(best_pruned_model, f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}_{pruned_model_accuracy}.pth')

titel_append=f'of {pruning_type} based Pruned {select_model.title()} model'
save_path=f'{basedir}/checkpoint/{select_model}/{pruning_type}/{select_model}_cifar_{pruning_type}'

plot_accuracy(accuracies,titel_append=titel_append,save_path=save_path+'_acc.png' )
plot_loss(train_losses,test_losses,titel_append=titel_append,save_path=save_path+'_loss.png')

In [ ]:
best_pruned_model=pruned_model
best_pruned_model.cuda()

In [ ]:

num_finetune_epochs=300
optimizer = torch.optim.SGD(best_pruned_model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

pruned_model_accuracy,best_pruned_model,accuracies,train_losses,test_losses=TrainingPrunned(best_pruned_model,train_dataloader,test_dataloader,criterion, optimizer, pruner,scheduler=scheduler,num_finetune_epochs=num_finetune_epochs,isCallback=isCallback)
